# Get list of entrants

In [98]:
tournament_slug = 'ngpr'
tournament_slug = 'aav'
event_name = 'Melee Redemption'
event_name = 'Melee Singles'

In [99]:
import json
from gql import gql, Client
from gql.transport.requests import RequestsHTTPTransport

In [100]:
%%bash
source ~/.bashrc

In [101]:
import os

auth_token = os.environ['SMASHGG_TOKEN']
api_version = 'alpha'

In [102]:
transport = RequestsHTTPTransport(
    url=f'https://api.start.gg/gql/{api_version}',
    headers={'Authorization': f'Bearer {auth_token}'},
    use_json=True,
)

client = Client(transport=transport, fetch_schema_from_transport=False)


In [103]:
# Define the API endpoint and query for tournament details
query = gql("""
query TournamentQuery($slug: String!) {
    tournament(slug: $slug) {
        id
        name
        city
        state
        countryCode
        startAt
        endAt
        events {
            id
            name
            numEntrants
        }
    }
}
""")

variables = {"slug": tournament_slug}

# Execute the query
try:
        tournament_result = client.execute(query, variable_values=variables)
        print(json.dumps(tournament_result, indent=2))
except Exception as e:
        print("Error fetching tournament details:", e)

{
  "tournament": {
    "id": 804685,
    "name": "Allston Allstars V - Tom's Birthday Bash!",
    "city": "Boston",
    "state": 1,
    "countryCode": "US",
    "startAt": 1754157600,
    "endAt": 1754190000,
    "events": [
      {
        "id": 1410041,
        "name": "Melee Singles",
        "numEntrants": 96
      },
      {
        "id": 1410043,
        "name": "Project+ Singles",
        "numEntrants": 20
      },
      {
        "id": 1410221,
        "name": "Melee Crews (5v5)",
        "numEntrants": 14
      },
      {
        "id": 1410048,
        "name": "Project+ Doubles",
        "numEntrants": 2
      },
      {
        "id": 1410042,
        "name": "Melee Redemption",
        "numEntrants": 0
      }
    ]
  }
}


In [104]:
# Extract the event id for "Melee Singles"
events = tournament_result['tournament']['events']
melee_singles_id = next((event['id'] for event in events if event_name in event['name']), None)
print(f"{event_name} Event ID: {melee_singles_id}")

Melee Singles Event ID: 1410041


In [105]:
# Query to get entrants for Melee Singles
entrants_query = gql("""
query EventEntrants($eventId: ID!) {
    event(id: $eventId) {
        entrants(query: {page: 1, perPage: 512}) {
            nodes {
                id
                name
                participants {
                    gamerTag
                }
            }
        }
    }
}
""")

entrants_variables = {"eventId": melee_singles_id}

try:
        entrants_result = client.execute(entrants_query, variable_values=entrants_variables)
        print(json.dumps(entrants_result, indent=2))
except Exception as e:
        print("Error fetching entrants:", e)

{
  "event": {
    "entrants": {
      "nodes": [
        {
          "id": 20855091,
          "name": "Sabriel",
          "participants": [
            {
              "gamerTag": "Sabriel"
            }
          ]
        },
        {
          "id": 20854858,
          "name": "sfy | LordTet",
          "participants": [
            {
              "gamerTag": "LordTet"
            }
          ]
        },
        {
          "id": 20852388,
          "name": "BU | BBron",
          "participants": [
            {
              "gamerTag": "BBron"
            }
          ]
        },
        {
          "id": 20852276,
          "name": "DarkCloud",
          "participants": [
            {
              "gamerTag": "DarkCloud"
            }
          ]
        },
        {
          "id": 20852205,
          "name": "MattDotZeb",
          "participants": [
            {
              "gamerTag": "MattDotZeb"
            }
          ]
        },
        {
          "id": 2085191

In [106]:
import polars as pl

# Extract entrant data and flatten participants' gamerTags

entrant_nodes = entrants_result['event']['entrants']['nodes']
data = []
for entrant in entrant_nodes:
    entrant_id = entrant['id']
    entrant_name = entrant['name']
    # There may be multiple participants per entrant; join their gamerTags with comma
    gamer_tags = [p['gamerTag'] for p in entrant.get('participants', [])]
    gamer_tag = ', '.join(gamer_tags)
    data.append({'id': entrant_id, 'name': entrant_name, 'gamertag': gamer_tag})

entrants = pl.DataFrame(data)
entrants

id,name,gamertag
i64,str,str
20855091,"""Sabriel""","""Sabriel"""
20854858,"""sfy | LordTet""","""LordTet"""
20852388,"""BU | BBron""","""BBron"""
20852276,"""DarkCloud""","""DarkCloud"""
20852205,"""MattDotZeb""","""MattDotZeb"""
20851915,"""Wishy""","""Wishy"""
20842180,"""Nico""","""Nico"""
20842018,"""Wolf""","""Wolf"""
20838313,"""PRIONBOIL""","""PRIONBOIL"""


In [107]:
entrant_gamertags = entrants.select(pl.col('gamertag').str.to_lowercase()).to_series().to_list()

# collect seeding

In [108]:
player_ratings = pl.read_parquet('data/player-ratings.parquet')

In [ ]:
pl.Config(tbl_rows=100)
entrant_ratings = (
    entrants
    .with_columns(pl.col('gamertag').str.to_lowercase())
    .join(
        player_ratings
            .with_columns(pl.col('tag').str.split(' | ').list.last().str.to_lowercase()),
        how='full',
        left_on='gamertag',
        right_on='tag',
    )
    #.filter(
    #    (pl.col('tag').str.to_lowercase() == pl.col('gamertag').str.to_lowercase()) |
    #    pl.col('gamertag').is_null()
    #)
    .filter(pl.col('gamertag').is_not_null())
    .sort('rating', descending=True)
    .group_by('id').first()
    .fill_null(0)
    .sort('rating', descending=True)
)
display(entrant_ratings)

id,name,gamertag,url,rating,tag
i64,str,str,str,f64,str
20564252,"""MATE | Kalvar""","""kalvar""","""/league/nemelee/player/C206957…",43.762813,"""kalvar"""
20564369,"""bonfire10""","""bonfire10""","""/league/nemelee/player/C9B8492…",42.003164,"""bonfire10"""
20564347,"""glock in my toyota""","""glock in my toyota""","""/league/nemelee/player/1FE504A…",39.884903,"""glock in my toyota"""
20567819,"""Project""","""project""","""/league/nemelee/player/55203EA…",39.53775,"""project"""
20564510,"""Ember""","""ember""","""/league/nemelee/player/10F6134…",39.526006,"""ember"""
20595089,"""Q""","""q""","""/league/nemelee/player/5C19CFD…",39.371221,"""q"""
20763728,"""YWRM | Q""","""q""","""/league/nemelee/player/5C19CFD…",39.371221,"""q"""
20564883,"""Hyouka | Nairial""","""nairial""","""/league/nemelee/player/39180F6…",38.907933,"""nairial"""
20602379,"""$G|MP | Bank""","""bank""","""/league/nemelee/player/E85EE0F…",38.823131,"""bank"""


In [110]:
entrant_seeding = (
    entrant_ratings
    .sort('rating', descending=True)
    .with_row_index('seed_num')
    .with_columns(pl.col('seed_num') + 1)
)
entrant_seeding

seed_num,id,name,gamertag,url,rating,tag
u32,i64,str,str,str,f64,str
1,20564252,"""MATE | Kalvar""","""kalvar""","""/league/nemelee/player/C206957…",43.762813,"""kalvar"""
2,20564369,"""bonfire10""","""bonfire10""","""/league/nemelee/player/C9B8492…",42.003164,"""bonfire10"""
3,20564347,"""glock in my toyota""","""glock in my toyota""","""/league/nemelee/player/1FE504A…",39.884903,"""glock in my toyota"""
4,20567819,"""Project""","""project""","""/league/nemelee/player/55203EA…",39.53775,"""project"""
5,20564510,"""Ember""","""ember""","""/league/nemelee/player/10F6134…",39.526006,"""ember"""
6,20595089,"""Q""","""q""","""/league/nemelee/player/5C19CFD…",39.371221,"""q"""
7,20763728,"""YWRM | Q""","""q""","""/league/nemelee/player/5C19CFD…",39.371221,"""q"""
8,20564883,"""Hyouka | Nairial""","""nairial""","""/league/nemelee/player/39180F6…",38.907933,"""nairial"""
9,20602379,"""$G|MP | Bank""","""bank""","""/league/nemelee/player/E85EE0F…",38.823131,"""bank"""


# Export seeding to start gg

In [51]:
# Query to get phase IDs for Melee Singles event
phases_query = gql("""
query EventPhases($eventId: ID!) {
    event(id: $eventId) {
        phases {
            id
            name
            numSeeds
        }
    }
}
""")

phases_variables = {"eventId": melee_singles_id}

try:
        phases_result = client.execute(phases_query, variable_values=phases_variables)
        print(json.dumps(phases_result, indent=2))
except Exception as e:
        print("Error fetching phases:", e)

phase_id = phases_result['event']['phases'][0]['id']
print(f'phase id: {phase_id}')

{
  "event": {
    "phases": [
      {
        "id": 2033907,
        "name": "Bracket",
        "numSeeds": 18
      }
    ]
  }
}
phase id: 2033907


In [52]:
# ...existing code...

# 1. Fetch seeds for the phase
seeds_query = gql("""
query PhaseSeeds($phaseId: ID!) {
  phase(id: $phaseId) {
    seeds(query: {perPage: 512}) {
      nodes {
        id
        entrant {
          id
        }
      }
    }
  }
}
""")
seeds_result = client.execute(seeds_query, variable_values={"phaseId": phase_id})
seed_nodes = seeds_result["phase"]["seeds"]["nodes"]
entrantid_to_seedid = {seed["entrant"]["id"]: seed["id"] for seed in seed_nodes}

# 2. Prepare seed mapping using correct seedId
seed_mapping = []
for row in entrant_seeding.iter_rows(named=True):
    entrant_id = int(row["id"])
    seed_id = entrantid_to_seedid.get(entrant_id)
    if seed_id:
        seed_mapping.append({
            "seedId": seed_id,
            "seedNum": row["seed_num"],
        })
    else:
        print(f"Warning: No seed found for entrant {entrant_id}")

print(f"Importing {len(seed_mapping)} seeds to phase {phase_id}...")

Importing 18 seeds to phase 2033907...


In [53]:
entrant_seeding = (
    entrant_seeding
    .with_columns(pl.col('id').map_elements(lambda x: entrantid_to_seedid.get(x, None)).alias('seed_id'))
)

/tmp/ipykernel_6676/1705157941.py:3: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  .with_columns(pl.col('id').map_elements(lambda x: entrantid_to_seedid.get(x, None)).alias('seed_id'))


In [54]:
# Prepare seed mapping from entrant_ratings for the phase
seed_mapping = []
for row in entrant_seeding.iter_rows(named=True):
    seed_mapping.append({
        "seedId": row["seed_id"],
        "seedNum": row['seed_num'],
    })

print(f"Importing {len(seed_mapping)} seeds to phase {phase_id}...")

mutation = gql("""
mutation UpdatePhaseSeeding($phaseId: ID!, $seedMapping: [UpdatePhaseSeedInfo]!) {
  updatePhaseSeeding(phaseId: $phaseId, seedMapping: $seedMapping) {
    id
  }
}
""")

params = {
    "phaseId": phase_id,
    "seedMapping": seed_mapping,
}

try:
    result = client.execute(mutation, variable_values=params)
    print('Success!')
    print(result)
except Exception as e:
    print('Error:', e)

Importing 18 seeds to phase 2033907...
Success!
{'updatePhaseSeeding': {'id': 2033907}}


# view previous tournaments by player

In [111]:
player = 'mattdotzeb'

In [112]:
from IPython.display import display, HTML

def display_side_by_side(*args, titles=('',)):
    html_str = ''
    if len(titles) > 0:
        html_str += '<div style="display:flex">'
    for df, title in zip(args, titles + ('',) * (len(args) - len(titles))):
        html_str += '<div style="margin-right:20px">'
        if title:
            html_str += f'<h2>{title}</h2>'
        html_str += df.to_html()
        html_str += '</div>'
    html_str += '</div>'
    display(HTML(html_str))

In [113]:
matches = pl.read_csv('data/matches-with-ratings.csv')

In [117]:
last_n_matches = (
    matches
    .filter(
        pl.col('winner').str.to_lowercase().str.contains(player.lower()) |
        pl.col('loser').str.to_lowercase().str.contains(player.lower())
    )
    .sort('tournament_date', 'encounter_id', descending=True)
    .head(40)
)
losses = (
    last_n_matches
    .filter(pl.col('loser').str.to_lowercase().str.contains(player.lower()))
    .select(
        pl.col('winner').alias('tag'),
        'winner_rating',
        'tournament_name',
        'tournament_date',
        'loser_score',
    )
    .sort('winner_rating', descending=False)
)
wins = (
    last_n_matches
    .filter(pl.col('winner').str.to_lowercase().str.contains(player.lower()))
    .select(
        pl.col('loser').alias('tag'),
        'loser_rating',
        'tournament_name',
        'tournament_date',
        'winner_score',
    )
    .sort('loser_rating', descending=True)
)
display_side_by_side(
    losses.to_pandas(),
    wins.to_pandas(),
    titles=('Losses', 'Wins')
)

,tag,winner_rating,tournament_name,tournament_date,loser_score
0,Snap,25.000000,New Game Plus Revival 6.13,2024-07-23,0
1,Younger,31.696917,New Game Plus Revival 4.16,2023-08-22,0
2,glock in my toyota,37.964520,New Game Plus Revival 4.16,2023-08-22,0
3,bonfire10,42.117923,New Game Plus Revival 6.13,2024-07-23,2
,tag,loser_rating,tournament_name,tournament_date,winner_score
0,Ant,35.113980,New Game Plus Revival 6.13,2024-07-23,2
1,PSI,33.540131,New Game Plus Revival 4.16,2023-08-22,2
2,Twisty,32.365027,New Game Plus Revival 6.13,2024-07-23,2
3,KoF | Thalia,31.063465,New Game Plus Revival 6.13,2024-07-23,3
4,NEM | Woodley,28.975606,New Game Plus Revival 4.16,2023-08-22,2
